# 09:Develop an AI Model to Analyze Social Media Sentiment Trends using Twitter or Reddit Dataset

# Step 1: Import Libraries

In [1]:
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Step 2: Load Dataset

In [2]:
data = pd.read_csv("Tweets.csv")

print(data.head())

             tweet_id airline_sentiment  airline_sentiment_confidence  \
0  570306133677760513           neutral                        1.0000   
1  570301130888122368          positive                        0.3486   
2  570301083672813571           neutral                        0.6837   
3  570301031407624196          negative                        1.0000   
4  570300817074462722          negative                        1.0000   

  negativereason  negativereason_confidence         airline  \
0            NaN                        NaN  Virgin America   
1            NaN                     0.0000  Virgin America   
2            NaN                        NaN  Virgin America   
3     Bad Flight                     0.7033  Virgin America   
4     Can't Tell                     1.0000  Virgin America   

  airline_sentiment_gold        name negativereason_gold  retweet_count  \
0                    NaN     cairdin                 NaN              0   
1                    NaN    jnar

# Step 3: Select Required Columns

In [3]:
data = data[['text', 'airline_sentiment']]

# Step 4: Convert Sentiment to Numbers

In [4]:
data['airline_sentiment'] = data['airline_sentiment'].map({
    'negative':0,
    'neutral':1,
    'positive':2
})

# Step 5: Split Dataset

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    data['text'],
    data['airline_sentiment'],
    test_size=0.2,
    random_state=42
)

# Step 6: Tokenization

In [6]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Step 7: Padding

In [7]:
X_train_pad = pad_sequences(X_train_seq, maxlen=100)
X_test_pad = pad_sequences(X_test_seq, maxlen=100)

# Step 8: Build Model

In [8]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Embedding(5000, 64, input_length=100),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(3, activation='softmax')
])

C:\Users\kambl\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


# Step 9: Compile Model

In [9]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Step 10: Train Model

In [10]:
model.fit(X_train_pad, y_train, epochs=5, validation_data=(X_test_pad, y_test))

Epoch 1/5
366/366 ━━━━━━━━━━━━━━━━━━━━ 33s 79ms/step - accuracy: 0.6731 - loss: 0.7866 - val_accuracy: 0.8016 - val_loss: 0.5063
Epoch 2/5
366/366 ━━━━━━━━━━━━━━━━━━━━ 35s 62ms/step - accuracy: 0.8303 - loss: 0.4332 - val_accuracy: 0.8132 - val_loss: 0.4890
Epoch 3/5
366/366 ━━━━━━━━━━━━━━━━━━━━ 23s 62ms/step - accuracy: 0.8872 - loss: 0.3149 - val_accuracy: 0.8019 - val_loss: 0.5073
Epoch 4/5
366/366 ━━━━━━━━━━━━━━━━━━━━ 24s 64ms/step - accuracy: 0.9080 - loss: 0.2506 - val_accuracy: 0.7954 - val_loss: 0.5515
Epoch 5/5
366/366 ━━━━━━━━━━━━━━━━━━━━ 20s 54ms/step - accuracy: 0.9252 - loss: 0.2089 - val_accuracy: 0.7958 - val_loss: 0.5816


# Step 11: Evaluate Model

In [11]:
loss, accuracy = model.evaluate(X_test_pad, y_test)
print("Accuracy:", accuracy)

92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.7942 - loss: 0.6012
Accuracy: 0.7957650423049927


# Step 12: Test Prediction

In [12]:
tweet = ["Flight was delayed and service was bad"]

seq = tokenizer.texts_to_sequences(tweet)
pad = pad_sequences(seq, maxlen=100)

pred = model.predict(pad)
result = pred.argmax()

if result == 0:
    print("Negative")
elif result == 1:
    print("Neutral")
else:
    print("Positive")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 276ms/step
Negative
